# Track A — Analytical Track Development & Validation

POC-12 — Lost Deal Reason & Recovery Intelligence

Approved track: Track A — Comparative Intelligence

Canonical data version: `phase3-v1.0.0`

Method version: `1.0.0`

In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().resolve()
if ROOT.name != "POC-12-Lost-Deal-Recovery-Phase3":
    ROOT = Path.cwd().resolve()

print("Workspace:", ROOT)
print("Canonical input:", ROOT / "data/canonical/intelligence_data.csv")

## 1. Canonical input

The analysis uses only `data/canonical/intelligence_data.csv`.

In [ ]:
import csv

input_path = ROOT / "data/canonical/intelligence_data.csv"
with input_path.open(newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

print("Records:", len(rows))
print("Data version:", sorted(set(r["data_version"] for r in rows)))

## 2. Approved method

Primary dimension: `category`

Minimum group size: 3

Measures: record count, total deal value, average deal value, percentage contribution, and deterministic rank.

In [ ]:
import subprocess

result = subprocess.run([sys.executable, "data-science/scripts/run_analytical_track.py"], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise SystemExit(result.returncode)

## 3. Validation

Validation independently recalculates the category results from the canonical CSV.

In [ ]:
result = subprocess.run([sys.executable, "data-science/scripts/validate_analytical_track.py"], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise SystemExit(result.returncode)

## 4. Weak-case review and export

In [ ]:
for script in ["review_weak_cases.py", "export_intelligence_results.py"]:
    result = subprocess.run([sys.executable, f"data-science/scripts/{script}"], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise SystemExit(result.returncode)

## 5. Evidence check

In [ ]:
output_files = [
    "data-science/outputs/comparative_intelligence.json",
    "data-science/outputs/validation_metrics.json",
    "data-science/outputs/weak_case_review.json",
    "data-science/outputs/intelligence_results.json",
    "data-science/outputs/intelligence_summary.json",
]

for name in output_files:
    path = ROOT / name
    print(name, "->", "PRESENT" if path.exists() else "MISSING")

validation = json.loads((ROOT / "data-science/outputs/validation_metrics.json").read_text())
print("Validation passed:", validation["passed"])
print("Contribution total:", validation["total_contribution_percent"], "%")

## Conclusion

The notebook executes the approved Track A method, validates the generated calculations, reviews weak cases, and regenerates the standard intelligence outputs from the canonical dataset.